# AI Portfolio Analyzer — Exploratory Data Analysis

This notebook walks through:
1. Downloading raw market data
2. Computing technical indicators
3. Visualising price, returns, volatility
4. Fetching and scoring news sentiment
5. Inspecting feature distributions
6. Running portfolio optimisation

> **Educational use only. Not financial advice.**

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.utils import load_config, set_seed, load_env
from src.dataset import download_ticker, build_features, get_feature_cols

load_env()
cfg = load_config('../configs/config.yaml')
set_seed(42)
print('Config loaded:', list(cfg.keys()))

In [ ]:
# ── 1. Download Data ────────────────────────────────────────────────────────
TICKER = 'AAPL'

raw = download_ticker(
    TICKER,
    start=cfg['data']['start_date'],
    end=cfg['data']['end_date'],
    raw_dir='../data/raw',
)
print(f'Raw shape: {raw.shape}')
raw.tail()

In [ ]:
# ── 2. Compute Features ────────────────────────────────────────────────────
df = build_features(raw, cfg)
print(f'Feature shape: {df.shape}')
print('Columns:', df.columns.tolist())
df.tail(3)

In [ ]:
# ── 3. Price + Moving Averages ─────────────────────────────────────────────
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Price & Moving Averages', 'Daily Volume'],
                    row_heights=[0.7, 0.3])

fig.add_trace(go.Candlestick(
    x=raw.index, open=raw['Open'], high=raw['High'],
    low=raw['Low'], close=raw['Close'], name='OHLC',
    increasing_line_color='#00d4aa', decreasing_line_color='#ff6b6b'
), row=1, col=1)

for w, color in zip([20, 50, 200], ['#ffd166', '#58a6ff', '#bc8cff']):
    fig.add_trace(go.Scatter(
        x=df.index, y=df[f'sma_{w}'],
        name=f'SMA {w}', line=dict(color=color, width=1.5, dash='dot')
    ), row=1, col=1)

fig.add_trace(go.Bar(
    x=raw.index, y=raw['Volume'], name='Volume',
    marker_color='#30363d'
), row=2, col=1)

fig.update_layout(
    title=f'{TICKER} — Price History',
    template='plotly_dark', height=600,
    xaxis_rangeslider_visible=False
)
fig.show()

In [ ]:
# ── 4. Returns Distribution ────────────────────────────────────────────────
ret = df['daily_return'].dropna()

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Daily Return Distribution', 'Rolling 20-Day Volatility'])

fig.add_trace(go.Histogram(
    x=ret, nbinsx=80, name='Daily Returns',
    marker_color='#00d4aa', opacity=0.75
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index, y=df['vol_20'] * 100,
    name='20d Vol (%)', line=dict(color='#ffd166', width=2)
), row=1, col=2)

fig.update_layout(template='plotly_dark', height=380, title=f'{TICKER} — Return Stats')
fig.show()

print(f"Mean daily return : {ret.mean()*100:.3f}%")
print(f"Std  daily return : {ret.std()*100:.3f}%")
print(f"Annual vol        : {ret.std()*np.sqrt(252)*100:.1f}%")
print(f"Skewness          : {ret.skew():.3f}")
print(f"Kurtosis          : {ret.kurtosis():.3f}")

In [ ]:
# ── 5. Technical Indicators ────────────────────────────────────────────────
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=['RSI (14)', 'MACD', 'Bollinger Bands'],
                    vertical_spacing=0.07)

# RSI
fig.add_trace(go.Scatter(x=df.index, y=df['rsi'], name='RSI',
                         line=dict(color='#bc8cff', width=1.5)), row=1, col=1)
fig.add_hline(y=70, line=dict(color='#ff6b6b', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=30, line=dict(color='#00d4aa', dash='dot', width=1), row=1, col=1)

# MACD
fig.add_trace(go.Scatter(x=df.index, y=df['macd'], name='MACD',
                         line=dict(color='#58a6ff', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['macd_signal'], name='Signal',
                         line=dict(color='#ffd166', width=1.5, dash='dot')), row=2, col=1)
fig.add_trace(go.Bar(x=df.index, y=df['macd_hist'], name='MACD Hist',
                     marker_color='#30363d'), row=2, col=1)

# Bollinger Bands
fig.add_trace(go.Scatter(x=df.index, y=raw['Close'], name='Close',
                         line=dict(color='#e6edf3', width=1.5)), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['bb_upper'], name='BB Upper',
                         line=dict(color='#ff6b6b', width=1, dash='dot')), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['bb_lower'], name='BB Lower',
                         line=dict(color='#00d4aa', width=1, dash='dot'),
                         fill='tonexty', fillcolor='rgba(0,212,170,0.05)'), row=3, col=1)

fig.update_layout(template='plotly_dark', height=700,
                  title=f'{TICKER} — Technical Indicators')
fig.show()

In [ ]:
# ── 6. Feature Correlation Heatmap ────────────────────────────────────────
feat_cols = get_feature_cols(use_sentiment=False)
feat_cols = [c for c in feat_cols if c in df.columns]
corr = df[feat_cols].dropna().corr()

# Show top 20 most correlated with daily_return
top20 = corr['daily_return'].abs().sort_values(ascending=False).head(20).index.tolist()
corr_sub = corr.loc[top20, top20]

fig = px.imshow(
    corr_sub,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title=f'{TICKER} — Top-20 Feature Correlation (by |corr| with daily_return)',
    template='plotly_dark',
    height=600,
)
fig.show()

In [ ]:
# ── 7. VaR / CVaR ─────────────────────────────────────────────────────────
from src.inference import compute_var_cvar

vc = compute_var_cvar(df['daily_return'].dropna(), confidence=0.95, horizon_days=21)
print('Risk Metrics (95% confidence):')
for k, v in vc.items():
    print(f'  {k:15s}: {v*100:+.3f}%')

In [ ]:
# ── 8. News Fetch + Sentiment ──────────────────────────────────────────────
from src.news import fetch_news
from src.sentiment import score_articles, aggregate_sentiment

articles = fetch_news(TICKER, cfg, max_articles=15)
print(f'Fetched {len(articles)} articles')

if articles:
    articles = score_articles(articles, cfg)
    sf = aggregate_sentiment(articles, cfg)
    print('\nSentiment features:')
    for k, v in sf.items():
        print(f'  {k:30s}: {v}')

    print('\nTop-3 articles:')
    for a in articles[:3]:
        sent = a.get('sentiment', {})
        print(f"  [{sent.get('label','?'):8s} {sent.get('score',0):+.2f}] {a['title'][:70]}")

In [ ]:
# ── 9. Portfolio Optimisation Preview ─────────────────────────────────────
from src.optimize import optimize_portfolio
import yfinance as yf

tickers = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'SPY']
prices = yf.download(tickers, start='2022-01-01', end='2024-12-31',
                     auto_adjust=True, progress=False)['Close']
prices.dropna(inplace=True)

result = optimize_portfolio(prices, cfg)
print(f"Method         : {result['method']}")
print(f"Expected Return: {result['expected_return']*100:.2f}%")
print(f"Expected Vol   : {result['expected_volatility']*100:.2f}%")
print(f"Sharpe Ratio   : {result['sharpe_ratio']:.3f}")
print('\nWeights:')
for t, w in result['weight_dict'].items():
    bar = '█' * int(w * 40)
    print(f'  {t:6s} {w*100:5.1f}%  {bar}')

In [ ]:
# ── 10. Multi-ticker Return Comparison ────────────────────────────────────
norm = prices / prices.iloc[0] * 100

fig = go.Figure()
colors = ['#00d4aa', '#58a6ff', '#ffd166', '#ff6b6b', '#bc8cff']
for t, c in zip(norm.columns, colors):
    fig.add_trace(go.Scatter(x=norm.index, y=norm[t], name=t,
                             line=dict(color=c, width=2)))

fig.update_layout(
    title='Normalised Performance (base = 100)',
    yaxis_title='Value (base 100)',
    template='plotly_dark',
    height=420,
)
fig.show()